In [5]:
#r "C:\Users\user\Desktop\Ainur\practice2026\task17\bin\Debug\net10.0\task17.dll"
#r "nuget:ScottPlot,5.0.54"

using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.IO;
using System.Linq;
using System.Threading;
using ScottPlot;
using task17;

public class WorkCommand : ICommand, IRepeatable
{
    private readonly int _maxSteps;
    private int _currentStep;
    
    public bool IsFinished() => _currentStep >= _maxSteps;
    
    public WorkCommand(int maxSteps)
    {
        _maxSteps = maxSteps;
    }
    
    public void Execute()
    {
        double sum = 0;
        for (int i = 0; i < 50000; i++)
            sum += Math.Sin(i) * Math.Cos(i);
        _currentStep++;
    }
}

Console.WriteLine("=== Сравнение: последовательно vs Round Robin ===\n");

int[] commandCounts = { 1, 2, 4, 8, 16 };
int stepsPerCommand = 5;

var sequentialTimes = new List<double>();
var roundRobinTimes = new List<double>();

foreach (int count in commandCounts)
{
    var sw1 = Stopwatch.StartNew();
    for (int j = 0; j < count; j++)
    {
        var cmd = new WorkCommand(stepsPerCommand);
        while (!cmd.IsFinished())
            cmd.Execute();
    }
    sw1.Stop();
    sequentialTimes.Add(sw1.Elapsed.TotalMilliseconds);

    var commands = new WorkCommand[count];
    for (int j = 0; j < count; j++)
        commands[j] = new WorkCommand(stepsPerCommand);

    var scheduler = new RoundRobinScheduler();
    foreach (var cmd in commands)
        scheduler.Add(cmd);

    var sw2 = Stopwatch.StartNew();
    while (scheduler.HasCommand())
    {
        var cmd = scheduler.Select();
        cmd.Execute();
        if (cmd is IRepeatable r && !r.IsFinished())
            scheduler.Add(cmd);
    }
    sw2.Stop();
    roundRobinTimes.Add(sw2.Elapsed.TotalMilliseconds);

    Console.WriteLine($"Команд: {count} | Последовательно: {sequentialTimes.Last():F2} мс | Round Robin: {roundRobinTimes.Last():F2} мс");
}

var xs = commandCounts.Select(c => (double)c).ToArray();

var plt = new Plot();
plt.Title("Время выполнения: Последовательно vs Round Robin");
plt.XLabel("Количество команд");
plt.YLabel("Время (мс)");

var s1 = plt.Add.Scatter(xs, sequentialTimes.ToArray());
s1.LegendText = "Последовательно";
s1.MarkerSize = 10;
s1.LineWidth = 3;

var s2 = plt.Add.Scatter(xs, roundRobinTimes.ToArray());
s2.LegendText = "Round Robin";
s2.MarkerSize = 10;
s2.LineWidth = 3;

plt.ShowLegend();
plt.SavePng("plot.png", 800, 600);
Console.WriteLine("\nГрафик: plot.png");

double avgOverhead = Enumerable.Range(0, commandCounts.Length).Select(i => (roundRobinTimes[i] - sequentialTimes[i]) / sequentialTimes[i] * 100).Average();

File.WriteAllText("report.txt", 
    $"Среднее время последовательно: {sequentialTimes.Average():F2} мс\n" +
    $"Среднее время Round Robin: {roundRobinTimes.Average():F2} мс\n" +
    $"Средние накладные расходы: {avgOverhead:F1}%\n" +
    $"График сохранен в plot.png");

Console.WriteLine("\nОтчет сохранен в report.txt");

Installed Packages ScottPlot, 5.0.54

=== Сравнение: последовательно vs Round Robin ===

Команд: 1 | Последовательно: 9.41 мс | Round Robin: 8.00 мс
Команд: 2 | Последовательно: 14.01 мс | Round Robin: 13.01 мс
Команд: 4 | Последовательно: 24.67 мс | Round Robin: 25.12 мс
Команд: 8 | Последовательно: 54.41 мс | Round Robin: 51.73 мс
Команд: 16 | Последовательно: 96.36 мс | Round Robin: 103.76 мс

График: plot.png

Отчет сохранен в report.txt
